# 01 – Scraping books.toscrape.com (Fiction & Nonfiction)

In [1]:
import os
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
CATEGORIES = {
    "Fiction": "https://books.toscrape.com/catalogue/category/books/fiction_10/index.html",
    "Nonfiction": "https://books.toscrape.com/catalogue/category/books/nonfiction_13/index.html",
}

IMG_DIR = "data/images"
os.makedirs(IMG_DIR, exist_ok=True)

In [3]:
def get_soup(url):
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"  # otherwise £ shows up as Â£
    time.sleep(0.5)  # short pause so we don't hammer the site
    return BeautifulSoup(resp.text, "html.parser")

In [4]:
soup = get_soup("https://books.toscrape.com/catalogue/category/books/fiction_10/index.html")
print(soup.title.text.strip())

Fiction | 
     Books to Scrape - Sandbox


## 1. Collect book links from each category

Each category page lists 20 books. Nonfiction has 110 books, so it spans several pages,
linked by a "next" button. We follow that button until there is none.

In [5]:
def get_book_links(category_url):
    links = []
    url = category_url
    while url:
        soup = get_soup(url)
        for a in soup.select("article.product_pod h3 a"):
            # the links are relative (../../book/index.html), urljoin makes them full URLs
            links.append(urljoin(url, a["href"]))

        next_link = soup.select_one("li.next a")
        if next_link:
            url = urljoin(url, next_link["href"])
        else:
            url = None
    return links

In [6]:
book_links = {}
for category, url in CATEGORIES.items():
    book_links[category] = get_book_links(url)
    print(category, len(book_links[category]))

Fiction 65
Nonfiction 110


## 2. Scrape a single book page

Selectors are limited to `div.product_main`, because the page also has a "recently viewed"
section with other books' prices and ratings. Values are stored raw (e.g. `"£54.23"`, `"Three"`)
and converted in the cleaning notebook.

In [7]:
def scrape_book(url, category):
    soup = get_soup(url)
    main = soup.select_one("div.product_main")

    # The rating is only stored as a CSS class, e.g. <p class="star-rating Three">
    rating = main.select_one("p.star-rating")["class"][1]

    # The description is the first <p> after the "Product Description" header
    desc_header = soup.select_one("#product_description")
    if desc_header:
        description = desc_header.find_next_sibling("p").text.strip()
    else:
        description = ""

    return {
        "title": main.h1.text.strip(),
        "category": category,
        "price": main.select_one("p.price_color").text.strip(),
        "rating": rating,
        "availability": main.select_one("p.availability").text.strip(),
        "description": description,
        # find the "UPC" row in the info table and take the value next to it
        "upc": soup.find("th", string="UPC").find_next("td").text,
        "url": url,
        "image_url": urljoin(url, soup.select_one("#product_gallery img")["src"]),
    }

In [8]:
scrape_book(book_links["Fiction"][0], "Fiction")

{'title': 'Soumission',
 'category': 'Fiction',
 'price': '£50.10',
 'rating': 'One',
 'availability': 'In stock (20 available)',
 'description': 'Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, s Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, se développe comme un mauvais rêve.Le talent de l’auteur, sa force visionnaire nous entraînent sur un t

## 3. Download cover images

Images are saved as `<UPC>.jpg`. The UPC is unique per book, so it links each image to its row.
Already downloaded images are skipped, so the notebook can be re-run safely.

In [9]:
def download_image(img_url, upc):
    path = f"{IMG_DIR}/{upc}.jpg"
    if not os.path.exists(path):  # skip images we already have
        resp = requests.get(img_url, timeout=10)
        resp.raise_for_status()
        with open(path, "wb") as f:
            f.write(resp.content)
        time.sleep(0.5)
    return path

## 4. Scrape everything and save to CSV

If one page fails, the error is printed and the loop continues with the next book.

In [10]:
books = []
for category, links in book_links.items():
    for url in links:
        try:
            book = scrape_book(url, category)
            book["image_path"] = download_image(book["image_url"], book["upc"])
            books.append(book)
        except Exception as e:
            print(f"Failed: {url} ({e})")
    print(f"{category}: done")

df = pd.DataFrame(books)
df.to_csv("data/books.csv", index=False, encoding="utf-8")
print(df.shape)
df.head()

Fiction: done
Nonfiction: done
(175, 10)


,title,category,price,rating,availability,description,upc,url,image_url,image_path
0,Soumission,Fiction,£50.10,One,In stock (20 available),"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/ee/cf/e...,data/images/6957f44c3847a760.jpg
1,Private Paris (Private #10),Fiction,£47.61,Five,In stock (17 available),Paris is burning--and only Private's Jack Morg...,b12b89017878a60d,https://books.toscrape.com/catalogue/private-p...,https://books.toscrape.com/media/cache/05/87/0...,data/images/b12b89017878a60d.jpg
2,"We Love You, Charlie Freeman",Fiction,£50.27,Five,In stock (16 available),"The Freeman family--Charles, Laurel, and their...",8d455c7539795d2a,https://books.toscrape.com/catalogue/we-love-y...,https://books.toscrape.com/media/cache/87/a2/8...,data/images/8d455c7539795d2a.jpg
3,Thirst,Fiction,£17.27,Five,In stock (16 available),"On a searing summer Friday, Eddie Chapman has ...",709822d0b5bcb7f4,https://books.toscrape.com/catalogue/thirst_94...,https://books.toscrape.com/media/cache/8c/54/8...,data/images/709822d0b5bcb7f4.jpg
4,The Murder That Never Was (Forensic Instincts #5),Fiction,£54.11,Three,In stock (16 available),"Given the opportunity, would you assume someon...",d01ac97e2b8947c2,https://books.toscrape.com/catalogue/the-murde...,https://books.toscrape.com/media/cache/29/e0/2...,data/images/d01ac97e2b8947c2.jpg


In [11]:
df["category"].value_counts()

category
Nonfiction    110
Fiction        65
Name: count, dtype: int64